In [ ]:
import sys
import os

# Dictionnaire des Opcodes
OPCODES = {
    "INIT":  "0",
    "LOAD":  "1",
    "STORE": "2",
    "JUMP":  "3",
    "JZ":    "4",
    "JN":    "5",
    "JP":    "6",
    "ADD":   "7",
    "SUB":   "8",
    "MULT":  "9",
    "AND":   "A",
    "NOT":   "B",
    "XOR":   "C",
    "SHL":   "D",
    "SHR":   "E",
    "HALT":  "F"
}

def parse_operand(operand_str):
    """Convertit l'opérande (décimal ou hexadécimal) en entier 12 bits."""
    if not operand_str:
        return 0
    
    try:
        if operand_str.lower().startswith('x'):
            # Formatage hexadécimal
            val = int(operand_str[1:], 16)
        elif operand_str.lower().startswith('0x'):
            val = int(operand_str[2:], 16)
        else:
            # Formatage décimal
            val = int(operand_str)
            
        # MASK sur 12 bits 
        return val & 0xFFF
    except ValueError:
        print(f"Erreur de syntaxe avec l'opérande : {operand_str}")
        sys.exit(1)

def compile_bolt16(input_file, txt_out_file):
    if not os.path.exists(input_file):
        print(f"Erreur : Le fichier {input_file} n'existe pas.")
        return

    output_lines = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    instruction_count = 1  # L'index mémoire commence à 1 

    for original_line in lines:

        line = original_line.strip()
        
        if not line or line.startswith(';') or line.startswith('#') or line.startswith('--'):
            continue
            
        # Séparation de l'instruction et de l'opcode
        parts = line.split()
        instruction = parts[0].upper()
        

        if instruction == "SUBSTARCT": instruction = "SUB"
        if instruction == "MULTIPLICATION": instruction = "MULT"
        if instruction == "SHITF_LEFT": instruction = "SHL"
        if instruction == "SHITF_RIGHT": instruction = "SHR"
        
        if instruction not in OPCODES:
            print(f"Erreur : Instruction inconnue '{instruction}' à la ligne : {line}")
            continue
            
        opcode_hex = OPCODES[instruction]
        
        operand_str = parts[1] if len(parts) > 1 else "0"
        
        if instruction in ["HALT", "SHL", "SHR", "NOT"]:
            operand_str = "x000"
            
        operand_val = parse_operand(operand_str)
        
        # Formatage hexadécimal 
        operand_hex = f"{operand_val:03X}"
        
        # Génération de la ligne formatée
        # Exemple: MEM(1) <= x"0" & x"005";        --  INIT 5
        formatted_line = f'MEM({instruction_count}) <= x"{opcode_hex}" & x"{operand_hex}";        --  {line}'
        output_lines.append(formatted_line)
        
        instruction_count += 1

    # Écriture dans le fichier de sortie
    with open(txt_out_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(output_lines) + '\n')
        
    print(f"Compilation réussie ! {instruction_count - 1} instructions traitées.")
    print(f" -> Fichier généré : {txt_out_file}")

if __name__ == "__main__":
    input_asm = "program.txt"
    out_txt = "rom_init.txt"
    
    args = [arg for arg in sys.argv[1:] if not arg.startswith('-')]
    
    if len(args) > 0:
        input_asm = args[0]
    if len(args) > 1:
        out_txt = args[1]
        
    # Création d'un fichier d'exemple s'il n'existe pas
    if not os.path.exists(input_asm):
        print(f"Fichier '{input_asm}' introuvable. Création d'un fichier d'exemple...")
        with open(input_asm, 'w') as f:
            f.write("INIT 5\nSTORE x001\nJUMP 7\nINIT 1\nINIT 1\nINIT 1\nINIT 1\nSUB x001\nHALT\n")
            
    compile_bolt16(input_asm, out_txt)

Compilation réussie ! 9 instructions traitées.
 -> Fichier VHDL généré      : rom_init.txt
 -> Fichier Bitstream généré : bitstream.txt
